# Chapter 3 Student Colab Notebook: Determinants

**Course:** MATH 3260 - Linear Algebra  
**Sections:** 3.1-3.4  
**Audience:** Student version

This notebook is the complete interactive teaching companion for Chapter 3. It combines theory, original worked examples, detailed determinant calculations, geometric interpretations, applications, visualizations, computational verification, section worksheets, and complete solutions.

> **How to use this student notebook**
>
> Read the complete theory and worked examples, run the demonstration code in order, and use each section worksheet for practice. Complete worksheet solutions are distributed separately. The notebook intentionally retains the chapter's full applications and visualizations.

## Chapter map

1. **Section 3.1:** The Determinant of a Matrix  
2. **Section 3.2:** Determinants and Elementary Operations  
3. **Section 3.3:** Properties of Determinants  
4. **Section 3.4:** Applications of Determinants  

Each section ends with a terminology table, a worksheet, and complete instructor solutions.

In [ ]:
# Core imports used throughout Chapter 3
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from IPython.display import display, Math, Markdown

sp.init_printing()


def show_matrix(M, label=None):
    """Display a SymPy matrix with an optional LaTeX label."""
    M = sp.Matrix(M)
    if label:
        display(Math(rf"{label}={sp.latex(M)}"))
    else:
        display(M)
    return M


def det2_steps(M, label='A'):
    """Show the complete 2-by-2 determinant calculation."""
    M = sp.Matrix(M)
    if M.shape != (2, 2):
        raise ValueError('M must be 2 by 2.')
    a, b, c, d = M[0,0], M[0,1], M[1,0], M[1,1]
    display(Math(rf"{label}={sp.latex(M)}"))
    display(Math(rf"\det({label})=({sp.latex(a)})({sp.latex(d)})-({sp.latex(b)})({sp.latex(c)})={sp.latex(M.det())}"))
    return M.det()


def minor_matrix(A, i, j):
    """Return the submatrix after deleting row i and column j (0-based)."""
    A = sp.Matrix(A)
    return A.minor_submatrix(i, j)


def all_minors_and_cofactors(A):
    """Return the minor matrix and cofactor matrix."""
    A = sp.Matrix(A)
    n = A.rows
    minors = sp.zeros(n)
    cofactors = sp.zeros(n)
    for i in range(n):
        for j in range(n):
            minors[i,j] = A.minor_submatrix(i,j).det()
            cofactors[i,j] = (-1)**(i+j) * minors[i,j]
    return minors, cofactors


def triangle_area(points):
    """Area of a triangle in the xy-plane from three points."""
    M = sp.Matrix([[x, y, 1] for x, y in points])
    return sp.Abs(M.det()) / 2


def tetrahedron_volume(points):
    """Volume of a tetrahedron from four 3D points."""
    M = sp.Matrix([[x, y, z, 1] for x, y, z in points])
    return sp.Abs(M.det()) / 6

print('Chapter 3 setup complete.')

> **Example policy**
>
> All worked examples and numerical values are original. The chapter uses personal-interest themes selectively, including the Spurs, hiking, travel, badminton, driving, Backstreet Boys, and R&B music. Any prices, route coordinates, or datasets are fictional teaching models unless explicitly identified otherwise.

# Section 3.1 - The Determinant of a Matrix

## Learning objectives

By the end of this section, students should be able to:

- compute the determinant of a $2\times2$ matrix;
- interpret a determinant as signed area scaling;
- find minors and cofactors;
- use cofactor expansion along any row or column;
- select an efficient expansion row or column;
- evaluate determinants of triangular matrices.

### Definition: determinant of a 2 by 2 matrix

> For $A=\begin{bmatrix}a&b\\c&d\end{bmatrix}$, the determinant is $\det(A)=ad-bc$. The notation $|A|$ is also used, but it should not be confused with absolute value.

## Original example: three $2\times2$ determinants

Consider

$$
A=\begin{bmatrix}9&1\\20&21\end{bmatrix},
\qquad
B=\begin{bmatrix}4&6\\2&3\end{bmatrix},
\qquad
C=\begin{bmatrix}-2&5\\7&1\end{bmatrix}.
$$

For $A$,

$$
\det(A)=9(21)-1(20)=189-20=169.
$$

For $B$,

$$
\det(B)=4(3)-6(2)=12-12=0.
$$

For $C$,

$$
\det(C)=(-2)(1)-5(7)=-2-35=-37.
$$

A zero determinant signals that the corresponding linear transformation collapses area and is not invertible.

In [ ]:
A31a = sp.Matrix([[9,1],[20,21]])
B31a = sp.Matrix([[4,6],[2,3]])
C31a = sp.Matrix([[-2,5],[7,1]])

det2_steps(A31a, 'A')
det2_steps(B31a, 'B')
det2_steps(C31a, 'C')

## Geometric meaning: signed area scaling

The columns of a $2\times2$ matrix determine a parallelogram. Its area is

$$
|\det(A)|.
$$

The sign records orientation:

- $\det(A)>0$: orientation is preserved;
- $\det(A)<0$: orientation is reversed;
- $\det(A)=0$: the parallelogram collapses to a line or point.

In [ ]:
# Visualize the unit square and its image under a 2-by-2 matrix.
def plot_area_scaling(A, title='Determinant as area scaling'):
    A = np.array(A, dtype=float)
    square = np.array([[0,0],[1,0],[1,1],[0,1],[0,0]])
    transformed = square @ A.T

    plt.figure(figsize=(7,6))
    plt.plot(square[:,0], square[:,1], marker='o', label='Unit square')
    plt.plot(transformed[:,0], transformed[:,1], marker='o', label='Transformed parallelogram')
    plt.axhline(0, linewidth=0.8)
    plt.axvline(0, linewidth=0.8)
    plt.grid(alpha=0.3)
    plt.axis('equal')
    plt.title(f"{title}\nArea scale = |det(A)| = {abs(np.linalg.det(A)):.2f}")
    plt.legend()
    plt.show()

plot_area_scaling([[2,1],[1,3]])

### Definition: minor and cofactor

> For a square matrix $A$, the minor $M_{ij}$ is the determinant obtained after deleting row $i$ and column $j$. The cofactor is $C_{ij}=(-1)^{i+j}M_{ij}$. The cofactor signs alternate in a checkerboard pattern.

### Why minors and cofactors are introduced

A $3\times3$ determinant is reduced to several $2\times2$ determinants. A **minor** is obtained by deleting one row and one column; a **cofactor** adds the alternating sign pattern. These ideas reappear later in the adjugate formula and Cramer's Rule.


The $3\times3$ cofactor sign pattern is

$$
\begin{bmatrix}
+&-&+\\
-&+&-\\
+&-&+
\end{bmatrix}.
$$

## Original example: all minors and cofactors

Let

$$
H=\begin{bmatrix}
2&1&0\\
-1&3&2\\
4&0&1
\end{bmatrix}.
$$

To find $M_{12}$, delete row 1 and column 2:

$$
M_{12}=\det\begin{bmatrix}-1&2\\4&1\end{bmatrix}
=(-1)(1)-2(4)=-9.
$$

Because $(-1)^{1+2}=-1$,

$$
C_{12}=-M_{12}=9.
$$

The complete minor matrix is

$$
[M_{ij}]=
\begin{bmatrix}
3&-9&-12\\
1&2&-4\\
2&4&7
\end{bmatrix},
$$

and the cofactor matrix is

$$
[C_{ij}]=
\begin{bmatrix}
3&9&-12\\
-1&2&4\\
2&-4&7
\end{bmatrix}.
$$

In [ ]:
H = sp.Matrix([[2,1,0],[-1,3,2],[4,0,1]])
minors_H, cofactors_H = all_minors_and_cofactors(H)
show_matrix(H, 'H')
show_matrix(minors_H, '[M_{ij}]')
show_matrix(cofactors_H, '[C_{ij}]')

## Cofactor expansion

For an $n\times n$ matrix, expansion along row $i$ is

$$
\det(A)=a_{i1}C_{i1}+a_{i2}C_{i2}+\cdots+a_{in}C_{in}.
$$

Expansion along column $j$ is

$$
\det(A)=a_{1j}C_{1j}+a_{2j}C_{2j}+\cdots+a_{nj}C_{nj}.
$$

The determinant is the same regardless of the selected row or column. Choose a row or column containing many zeros.

### Detailed expansion of $H$ along the first row

Because the third entry in the first row is zero,

$$
\det(H)=2C_{11}+1C_{12}+0C_{13}.
$$

We already found $C_{11}=3$ and $C_{12}=9$, so

$$
\det(H)=2(3)+1(9)+0(-12)=15.
$$

The key pattern is

$$
\det(H)=h_{11}\det(M_{11})-h_{12}\det(M_{12})+h_{13}\det(M_{13}),
$$

and every $2\times2$ minor then uses $ad-bc$. A common $3\times3$ error is forgetting the negative sign on the middle cofactor.


In [ ]:
print('det(H) =', H.det())
# Verify expansions along every row and every column.
for i in range(3):
    row_sum = sum(H[i,j] * H.cofactor(i,j) for j in range(3))
    print(f'Expansion along row {i+1}:', sp.simplify(row_sum))
for j in range(3):
    col_sum = sum(H[i,j] * H.cofactor(i,j) for i in range(3))
    print(f'Expansion along column {j+1}:', sp.simplify(col_sum))

## Original $4\times4$ example: exploit a sparse column

Let

$$
Q=
\begin{bmatrix}
2&1&0&3\\
0&-1&0&4\\
5&2&7&1\\
1&0&0&2
\end{bmatrix}.
$$

The third column has only one nonzero entry, $q_{33}=7$. Therefore,

$$
\det(Q)=7C_{33}.
$$

Since $(-1)^{3+3}=1$,

$$
C_{33}=\det\begin{bmatrix}
2&1&3\\
0&-1&4\\
1&0&2
\end{bmatrix}.
$$

Expand this $3\times3$ determinant along the second row. The middle entry is $-1$:

$$
(-1)(+1)\det\begin{bmatrix}2&3\\1&2\end{bmatrix}
=(-1)(4-3)=-1.
$$

The last entry is $4$ and its cofactor sign is negative:

$$
4(-1)\det\begin{bmatrix}2&1\\1&0\end{bmatrix}
=4(-1)(-1)=4.
$$

Therefore,

$$
C_{33}=-1+4=3.
$$

Hence

$$
\det(Q)=7(3)=21.
$$

In [ ]:
Q = sp.Matrix([[2,1,0,3],[0,-1,0,4],[5,2,7,1],[1,0,0,2]])
show_matrix(Q, 'Q')
print('det(Q) =', Q.det())
print('Minor M_33 =', Q.minor_submatrix(2,2).det())

### Theorem: determinant of a triangular matrix

> If $A$ is upper triangular or lower triangular, then $\det(A)$ is the product of the diagonal entries.

For

$$
T=\begin{bmatrix}
3&0&0&0\\
-2&5&0&0\\
1&4&-2&0\\
7&1&6&4
\end{bmatrix},
$$

we have

$$
\det(T)=3(5)(-2)(4)=-120.
$$

In [ ]:
T = sp.Matrix([[3,0,0,0],[-2,5,0,0],[1,4,-2,0],[7,1,6,4]])
show_matrix(T, 'T')
print('Product of diagonal entries =', sp.prod(T[i,i] for i in range(4)))
print('det(T) =', T.det())

> **Common mistakes**
>
> Do not apply the $ad-bc$ rule to matrices larger than $2\times2$. Do not forget the alternating cofactor signs. During cofactor expansion, multiply each entry by its cofactor, not merely by its minor.

## Definitions and terminology

| Term | Meaning |
|---|---|
| Determinant | Scalar associated with a square matrix |
| Signed area scale | Geometric interpretation of a $2\times2$ determinant |
| Minor $M_{ij}$ | Determinant after deleting row $i$ and column $j$ |
| Cofactor $C_{ij}$ | $(-1)^{i+j}M_{ij}$ |
| Cofactor expansion | Sum of entries times cofactors along a row or column |
| Upper triangular | Zeros below the main diagonal |
| Lower triangular | Zeros above the main diagonal |
| Diagonal matrix | Zeros above and below the main diagonal |

## Section 3.1 worksheet

1. Compute

$$
\det\begin{bmatrix}7&2\\3&5\end{bmatrix}.
$$

2. For

$$
A=\begin{bmatrix}1&2&0\\3&-1&4\\2&1&5\end{bmatrix},
$$

find all minors and cofactors.

3. Use cofactor expansion to evaluate

$$
\det\begin{bmatrix}4&0&2\\1&3&0\\0&5&1\end{bmatrix}.
$$

4. Find the determinant of

$$
\begin{bmatrix}2&0&0&0\\-1&3&0&0\\4&5&-2&0\\7&1&6&5\end{bmatrix}.
$$

5. Explain why a $2\times2$ matrix with determinant zero collapses area.

# Section 3.2 - Determinants and Elementary Operations

## Learning objectives

Students should be able to:

- track determinant changes under row operations;
- track determinant changes under column operations;
- evaluate determinants efficiently by reducing to triangular form;
- recognize structural conditions that force a zero determinant;
- connect a zero determinant with redundant information.

### Theorem: elementary row operations

> Interchanging two rows changes the sign of the determinant. Adding a multiple of one row to another leaves the determinant unchanged. Multiplying one row by $c$ multiplies the determinant by $c$. The same rules hold for column operations.

## Detailed example: triangularization without changing the determinant

Let

$$
A=\begin{bmatrix}
2&1&-1\\
4&3&0\\
-2&2&5
\end{bmatrix}.
$$

Start with

$$
\det(A)=
\det\begin{bmatrix}
2&1&-1\\
4&3&0\\
-2&2&5
\end{bmatrix}.
$$

Apply

$$
R_2\leftarrow R_2-2R_1.
$$

Adding a multiple of one row to another does not change the determinant:

$$
\det(A)=
\det\begin{bmatrix}
2&1&-1\\
0&1&2\\
-2&2&5
\end{bmatrix}.
$$

Next apply

$$
R_3\leftarrow R_3+R_1,
$$

giving

$$
\det(A)=
\det\begin{bmatrix}
2&1&-1\\
0&1&2\\
0&3&4
\end{bmatrix}.
$$

Finally apply

$$
R_3\leftarrow R_3-3R_2,
$$

giving

$$
\det(A)=
\det\begin{bmatrix}
2&1&-1\\
0&1&2\\
0&0&-2
\end{bmatrix}.
$$

The matrix is upper triangular, so

$$
\det(A)=2(1)(-2)=-4.
$$

In [ ]:
A32 = sp.Matrix([[2,1,-1],[4,3,0],[-2,2,5]])
A32_1 = A32.copy()
A32_1[1,:] = A32_1[1,:] - 2*A32_1[0,:]
A32_2 = A32_1.copy()
A32_2[2,:] = A32_2[2,:] + A32_2[0,:]
A32_3 = A32_2.copy()
A32_3[2,:] = A32_3[2,:] - 3*A32_3[1,:]

show_matrix(A32, 'A')
show_matrix(A32_1, 'R_2-2R_1')
show_matrix(A32_2, 'R_3+R_1')
show_matrix(A32_3, 'R_3-3R_2')
print('det(A) =', A32.det())

## Row interchange and row scaling

If $B$ is obtained by swapping two rows of $A$, then

$$
\det(B)=-\det(A).
$$

If $C$ is obtained by multiplying one row of $A$ by $5$, then

$$
\det(C)=5\det(A).
$$

If several row operations are used, keep a determinant accounting record. A common strategy is to avoid scaling rows until the final triangular matrix is reached.

### Determinant effect of the row operations

$$
\begin{array}{c|c}
R_i\leftarrow R_i+cR_j & \det(B)=\det(A)\\
R_i\leftrightarrow R_j & \det(B)=-\det(A)\\
R_i\leftarrow cR_i & \det(B)=c\,\det(A)
\end{array}
$$


In [ ]:
B32 = A32.copy()
B32.row_swap(0,1)
C32 = A32.copy()
C32[1,:] = 5*C32[1,:]
print('det(A) =', A32.det())
print('After one row swap:', B32.det())
print('After multiplying row 2 by 5:', C32.det())

## Column operations and zero determinants

Consider

$$
D=\begin{bmatrix}
1&2&3\\
4&8&5\\
2&4&7
\end{bmatrix}.
$$

The second column is twice the first column. Apply

$$
C_2\leftarrow C_2-2C_1.
$$

This column replacement does not change the determinant, and it creates a zero column:

$$
\det(D)=
\det\begin{bmatrix}
1&0&3\\
4&0&5\\
2&0&7
\end{bmatrix}=0.
$$

A zero determinant is also a dependence test: if rows or columns are linearly dependent, then $\det(A)=0$.


In [ ]:
D32 = sp.Matrix([[1,2,3],[4,8,5],[2,4,7]])
show_matrix(D32, 'D')
print('det(D) =', D32.det())
print('Column 2 - 2 Column 1 =', D32[:,1] - 2*D32[:,0])

> **Conditions that force a zero determinant**
>
> A square matrix has determinant zero if it has an all-zero row or column, two equal rows or columns, or one row or column that is a scalar multiple of another. More generally, any linear dependence among the rows or columns forces determinant zero.

## Application: redundant data features

Suppose columns of a data matrix represent three recorded features, but the second feature is exactly twice the first. Then the columns are linearly dependent. The determinant is zero, so the matrix cannot be inverted. In modeling terms, the second feature contributes no new independent information.

In [ ]:
# Explore determinant changes under simple row operations.
def determinant_operation_demo(A):
    A = sp.Matrix(A)
    print('Original determinant:', A.det())

    swap = A.copy(); swap.row_swap(0,1)
    print('After swapping rows 1 and 2:', swap.det())

    add = A.copy(); add[1,:] = add[1,:] + 3*add[0,:]
    print('After R2 <- R2 + 3R1:', add.det())

    scale = A.copy(); scale[0,:] = -2*scale[0,:]
    print('After R1 <- -2R1:', scale.det())

determinant_operation_demo([[2,1],[3,4]])

## Definitions and terminology

| Term | Meaning |
|---|---|
| Row interchange | Swap two rows; determinant changes sign |
| Row replacement | Add a multiple of one row to another; determinant unchanged |
| Row scaling | Multiply one row by $c$; determinant multiplied by $c$ |
| Column-equivalent | Related by elementary column operations |
| Zero determinant | Determinant equal to zero; matrix is singular |
| Redundant information | A row or column depends on others |

## Section 3.2 worksheet

1. Use row operations to find

$$
\det\begin{bmatrix}3&1&2\\6&5&1\\0&4&7\end{bmatrix}.
$$

2. Suppose $\det(A)=-6$. Find the determinant after each operation:

- interchange rows 1 and 2;
- multiply row 3 by $-3$;
- add $5$ times row 1 to row 2.

3. Explain why

$$
\det\begin{bmatrix}2&5&1\\4&10&2\\3&1&7\end{bmatrix}=0.
$$

4. Use a column operation to evaluate

$$
\det\begin{bmatrix}1&3&2\\2&6&5\\4&12&1\end{bmatrix}.
$$

5. A data matrix has a column equal to the sum of two other columns. What can you conclude about its determinant and invertibility?

# Section 3.3 - Properties of Determinants

## Learning objectives

Students should be able to:

- compute determinants of matrix products without multiplying first;
- use the scalar-multiple rule $\det(cA)=c^n\det(A)$;
- compute determinants of inverses and powers;
- connect nonzero determinant with invertibility;
- use $\det(A^T)=\det(A)$;
- interpret determinant rules through composition of transformations.

### Theorem: determinant of a product

> For square matrices of the same order, $\det(AB)=\det(A)\det(B)$. Matrix multiplication is generally not commutative, but determinants satisfy $\det(AB)=\det(BA)$.

## Original example: product property

Let

$$
A=\begin{bmatrix}2&1\\1&3\end{bmatrix},
\qquad
B=\begin{bmatrix}1&-1\\4&2\end{bmatrix}.
$$

Then

$$
\det(A)=2(3)-1(1)=5,
$$

and

$$
\det(B)=1(2)-(-1)(4)=6.
$$

Therefore,

$$
\det(AB)=\det(A)\det(B)=5(6)=30.
$$

Indeed,

$$
AB=\begin{bmatrix}6&0\\13&5\end{bmatrix},
$$

so

$$
\det(AB)=6(5)-0(13)=30.
$$

### Why $\det(AB)=\det(BA)$

$$
\det(AB)=\det(A)\det(B)=\det(B)\det(A)=\det(BA).
$$

This does **not** imply $AB=BA$.


In [ ]:
A33 = sp.Matrix([[2,1],[1,3]])
B33 = sp.Matrix([[1,-1],[4,2]])
show_matrix(A33, 'A')
show_matrix(B33, 'B')
show_matrix(A33*B33, 'AB')
print('det(A)det(B) =', A33.det()*B33.det())
print('det(AB) =', (A33*B33).det())

## Geometric interpretation of the product rule

If $B$ scales area by $\det(B)$ and then $A$ scales the resulting area by $\det(A)$, the combined transformation $AB$ scales area by the product

$$
\det(A)\det(B).
$$

This is why the determinant of a composition is multiplicative.

### Theorem: scalar multiples

> If $A$ is an $n\times n$ matrix, then $\det(cA)=c^n\det(A)$. Multiplying the entire matrix by $c$ multiplies each of its $n$ rows by $c$.

For the upper triangular matrix

$$
C=\begin{bmatrix}1&2&0\\0&3&1\\0&0&2\end{bmatrix},
$$

we have

$$
\det(C)=1(3)(2)=6.
$$

Because $C$ is $3\times3$,

$$
\det(2C)=2^3\det(C)=8(6)=48.
$$

In [ ]:
C33 = sp.Matrix([[1,2,0],[0,3,1],[0,0,2]])
print('det(C) =', C33.det())
print('det(2C) =', (2*C33).det())
print('2^3 det(C) =', 2**3*C33.det())

> **Invertibility and inverse determinants**
>
> A square matrix is invertible if and only if its determinant is nonzero. If $A$ is invertible, then $\det(A^{-1})=1/\det(A)$.

For

$$
A=\begin{bmatrix}2&1\\1&3\end{bmatrix},
$$

we found $\det(A)=5$. Therefore,

$$
\det(A^{-1})=\frac{1}{5}.
$$

This follows from

$$
1=\det(I)=\det(AA^{-1})=\det(A)\det(A^{-1}).
$$

In [ ]:
show_matrix(A33.inv(), 'A^{-1}')
print('det(A^{-1}) =', sp.factor(A33.inv().det()))
print('1/det(A) =', sp.Rational(1, A33.det()))

### A nontriangular scalar-multiple example

For

$$
A=\begin{bmatrix}1&2\\3&5\end{bmatrix},
\qquad \det(A)=-1,
$$

we have

$$
\det(4A)=-16=4^2\det(A),
$$

illustrating $\det(cA)=c^n\det(A)$ beyond the triangular case.


## Equivalent conditions for a nonsingular matrix

For an $n\times n$ matrix $A$, the following conditions are equivalent:

1. $A$ is invertible;
2. $A\mathbf{x}=\mathbf{b}$ has a unique solution for every $\mathbf{b}$;
3. $A\mathbf{x}=\mathbf{0}$ has only the trivial solution;
4. $A$ is row-equivalent to $I_n$;
5. $A$ is a product of elementary matrices;
6. $\det(A)\neq0$.

A single determinant calculation can therefore answer several structural questions.

These conditions are interchangeable tests for the same structure. For example, a pivot in every column implies invertibility, hence $\det(A)\ne0$, independent columns, and only the trivial solution to $A\mathbf x=\mathbf0$.


### Theorem: transpose

> For every square matrix, $\det(A^T)=\det(A)$. Transposition exchanges rows and columns but preserves the determinant.

Let

$$
R=\begin{bmatrix}2&0&1\\-1&3&2\\4&1&0\end{bmatrix}.
$$

A direct calculation gives

$$
\det(R)=-13.
$$

Its transpose is

$$
R^T=\begin{bmatrix}2&-1&4\\0&3&1\\1&2&0\end{bmatrix},
$$

and

$$
\det(R^T)=-13.
$$

In [ ]:
R33 = sp.Matrix([[2,0,1],[-1,3,2],[4,1,0]])
show_matrix(R33, 'R')
show_matrix(R33.T, 'R^T')
print('det(R) =', R33.det())
print('det(R^T) =', R33.T.det())

## Determinants of powers

Repeated use of the product theorem gives

$$
\det(A^k)=\det(A)^k
$$

for every positive integer $k$. For invertible $A$, the rule also extends to negative integer powers.

For the matrix $A$ above with $\det(A)=5$,

$$
\det(A^4)=5^4=625.
$$

In [ ]:
print('det(A^4) =', (A33**4).det())
print('det(A)^4 =', A33.det()**4)

## Definitions and terminology

| Term | Meaning |
|---|---|
| Product rule | $\det(AB)=\det(A)\det(B)$ |
| Scalar-multiple rule | $\det(cA)=c^n\det(A)$ for $n\times n$ $A$ |
| Nonsingular | Invertible; determinant nonzero |
| Singular | Not invertible; determinant zero |
| Inverse determinant | $\det(A^{-1})=1/\det(A)$ |
| Transpose determinant | $\det(A^T)=\det(A)$ |
| Determinant of a power | $\det(A^k)=\det(A)^k$ |

## Section 3.3 worksheet

1. Suppose $\det(A)=4$ and $\det(B)=-3$ for $3\times3$ matrices. Find:

- $\det(AB)$;
- $\det(BA)$;
- $\det(2A)$;
- $\det(B^{-1})$.

2. Find $\det(2AB^{-1})$ using the same information.

3. If $\det(C)=0$, state what you can conclude about $C^{-1}$ and the homogeneous system $C\mathbf{x}=\mathbf{0}$.

4. A two-dimensional transformation first scales area by $-2$ and then by $5$. What is the signed area scale of the composition?

5. If $\det(D)=-2$, find $\det((D^T)^3)$.

# Section 3.4 - Applications of Determinants

## Learning objectives

Students should be able to:

- form the cofactor matrix and adjugate;
- find an inverse using the adjugate formula;
- solve square systems using Cramer's Rule;
- compute triangle area and test collinearity;
- write a determinant equation for a line;
- compute tetrahedron volume and test coplanarity;
- write an equation of a plane through three points.

### Definition: adjugate

> The adjugate of $A$, denoted $\operatorname{adj}(A)$, is the transpose of the cofactor matrix. If $\det(A)\neq0$, then $A^{-1}=\frac{1}{\det(A)}\operatorname{adj}(A)$.

## Detailed adjugate example

Let

$$
M=\begin{bmatrix}
1&1&0\\
0&1&1\\
1&0&1
\end{bmatrix}.
$$

The cofactor matrix is

$$
C=\begin{bmatrix}
1&1&-1\\
-1&1&1\\
1&-1&1
\end{bmatrix}.
$$

Therefore,

$$
\operatorname{adj}(M)=C^T
=\begin{bmatrix}
1&-1&1\\
1&1&-1\\
-1&1&1
\end{bmatrix}.
$$

Since $\det(M)=2$,

$$
M^{-1}
=\frac12\operatorname{adj}(M)
=\frac12\begin{bmatrix}
1&-1&1\\
1&1&-1\\
-1&1&1
\end{bmatrix}.
$$

The adjugate is useful because

$$
A^{-1}=\frac1{\det(A)}\operatorname{adj}(A),
\qquad \det(A)\ne0.
$$

So the cofactor calculations from Section 3.1 directly build an inverse.


In [ ]:
M34 = sp.Matrix([[1,1,0],[0,1,1],[1,0,1]])
show_matrix(M34, 'M')
show_matrix(M34.cofactor_matrix(), 'C')
show_matrix(M34.adjugate(), r'\operatorname{adj}(M)')
show_matrix(M34.inv(), 'M^{-1}')
print('M * M^{-1} =')
display(M34*M34.inv())

### Cramer's Rule

> For $A\mathbf{x}=\mathbf{b}$ with $\det(A)\neq0$, replace column $i$ of $A$ with $\mathbf{b}$ to form $A_i$. Then $x_i=\det(A_i)/\det(A)$.

## Original application: concert merchandise prices

At a fictional Backstreet Boys fan event, let

- $p$ = poster price;
- $w$ = wristband price;
- $t$ = tour-book price.

Three bundle totals give

$$
\begin{cases}
p+w=29,\\
w+t=41,\\
p+t=30.
\end{cases}
$$

The coefficient matrix and constant vector are

$$
A=\begin{bmatrix}1&1&0\\0&1&1\\1&0&1\end{bmatrix},
\qquad
\mathbf{b}=\begin{bmatrix}29\\41\\30\end{bmatrix}.
$$

We have

$$
D=\det(A)=2.
$$

Replace the first column by $\mathbf{b}$:

$$
D_p=\det\begin{bmatrix}29&1&0\\41&1&1\\30&0&1\end{bmatrix}=18.
$$

Similarly,

$$
D_w=40,
\qquad
D_t=42.
$$

Therefore,

$$
p=\frac{18}{2}=9,
\qquad
w=\frac{40}{2}=20,
\qquad
t=\frac{42}{2}=21.
$$

In [ ]:
b34 = sp.Matrix([29,41,30])
variables = sp.symbols('p w t')
D = M34.det()
Ds=[]
for j in range(3):
    Aj=M34.copy()
    Aj[:,j]=b34
    Ds.append(Aj.det())
print('D =', D)
print('Replacement determinants =', Ds)
print('Solution =', [sp.Rational(value,D) for value in Ds])

## Area of a triangle and collinearity

For vertices $(x_1,y_1)$, $(x_2,y_2)$, and $(x_3,y_3)$,

$$
\text{Area}
=\frac12\left|
\det\begin{bmatrix}
x_1&y_1&1\\
x_2&y_2&1\\
x_3&y_3&1
\end{bmatrix}
\right|.
$$

### Hiking-map example

Three trail landmarks have coordinates

$$
(1,2),\qquad(7,5),\qquad(4,10).
$$

Then

$$
\det\begin{bmatrix}1&2&1\\7&5&1\\4&10&1\end{bmatrix}=39,
$$

so

$$
\text{Area}=\frac{|39|}{2}=\frac{39}{2}=19.5
$$

square map units. Three points are collinear exactly when this determinant is zero.

If the determinant-based area is zero, the triangle has zero area, so the three points are collinear.


In [ ]:
trail_points = [(1,2),(7,5),(4,10)]
print('Triangle area =', triangle_area(trail_points))

xs=[p[0] for p in trail_points]+[trail_points[0][0]]
ys=[p[1] for p in trail_points]+[trail_points[0][1]]
plt.figure(figsize=(6,5))
plt.plot(xs,ys,marker='o')
for label,(x,y) in zip(['A','B','C'],trail_points):
    plt.text(x+0.15,y+0.15,label)
plt.grid(alpha=0.3)
plt.axis('equal')
plt.title('Hiking-map triangle')
plt.xlabel('east-west coordinate')
plt.ylabel('north-south coordinate')
plt.show()

## Determinant equation of a line

A point $(x,y)$ lies on the line through $(x_1,y_1)$ and $(x_2,y_2)$ when

$$
\det\begin{bmatrix}
x&y&1\\
x_1&y_1&1\\
x_2&y_2&1
\end{bmatrix}=0.
$$

For travel-map points $(2,5)$ and $(8,17)$,

$$
\det\begin{bmatrix}
x&y&1\\2&5&1\\8&17&1
\end{bmatrix}=0.
$$

Expanding gives

$$
-12x+6y-6=0,
$$

or

$$
y=2x+1.
$$

In [ ]:
x,y=sp.symbols('x y')
line_det=sp.Matrix([[x,y,1],[2,5,1],[8,17,1]]).det()
print('Determinant equation expands to:', sp.expand(line_det))
print('Solved for y:', sp.solve(sp.Eq(line_det,0),y)[0])

## Volume of a tetrahedron

For four points in space, the tetrahedron volume is

$$
V=\frac16\left|
\det\begin{bmatrix}
x_1&y_1&z_1&1\\
x_2&y_2&z_2&1\\
x_3&y_3&z_3&1\\
x_4&y_4&z_4&1
\end{bmatrix}
\right|.
$$

For

$$
(0,0,0),\quad(2,0,0),\quad(0,3,0),\quad(0,0,5),
$$

we obtain

$$
V=\frac16|30|=5
$$

cubic units.

In [ ]:
tetra_points=[(0,0,0),(2,0,0),(0,3,0),(0,0,5)]
print('Tetrahedron volume =', tetrahedron_volume(tetra_points))

from mpl_toolkits.mplot3d.art3d import Poly3DCollection
verts=np.array(tetra_points,dtype=float)
faces=[[verts[0],verts[1],verts[2]],[verts[0],verts[1],verts[3]],[verts[0],verts[2],verts[3]],[verts[1],verts[2],verts[3]]]
fig=plt.figure(figsize=(7,6))
ax=fig.add_subplot(111,projection='3d')
poly=Poly3DCollection(faces,alpha=0.25,edgecolor='black')
ax.add_collection3d(poly)
ax.scatter(verts[:,0],verts[:,1],verts[:,2],s=45)
ax.set_xlim(0,2.5); ax.set_ylim(0,3.5); ax.set_zlim(0,5.5)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title('Tetrahedron with volume 5')
plt.show()

## Equation of a plane through three points

A point $(x,y,z)$ lies in the plane through three noncollinear points when

$$
\det\begin{bmatrix}
x&y&z&1\\
x_1&y_1&z_1&1\\
x_2&y_2&z_2&1\\
x_3&y_3&z_3&1
\end{bmatrix}=0.
$$

Use the points

$$
P_1=(1,0,2),\qquad P_2=(3,1,0),\qquad P_3=(0,2,1).
$$

Vectors in the plane are

$$
P_2-P_1=(2,1,-2),
\qquad
P_3-P_1=(-1,2,-1).
$$

Their cross product is

$$
(2,1,-2)\times(-1,2,-1)=(3,4,5).
$$

Therefore, the plane has equation

$$
3(x-1)+4(y-0)+5(z-2)=0,
$$

or

$$
3x+4y+5z=13.
$$

A fourth point is coplanar with these three exactly when it satisfies this equation, equivalently when the corresponding $4\times4$ determinant is zero.

In the example, $(3,4,5)$ is a **normal vector** to the plane.


In [ ]:
P1=sp.Matrix([1,0,2])
P2=sp.Matrix([3,1,0])
P3=sp.Matrix([0,2,1])
normal=(P2-P1).cross(P3-P1)
show_matrix(normal, 'n')
xx,yy,zz=sp.symbols('x y z')
plane_expr=sp.expand(normal.dot(sp.Matrix([xx,yy,zz])-P1))
print('Plane expression =', plane_expr)

# Plot the plane and the three points.
xgrid,ygrid=np.meshgrid(np.linspace(-1,4,30),np.linspace(-1,4,30))
zgrid=(13-3*xgrid-4*ygrid)/5
fig=plt.figure(figsize=(7,6))
ax=fig.add_subplot(111,projection='3d')
ax.plot_surface(xgrid,ygrid,zgrid,alpha=0.25)
pts=np.array([[1,0,2],[3,1,0],[0,2,1]],dtype=float)
ax.scatter(pts[:,0],pts[:,1],pts[:,2],s=55)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title('Plane through three points')
plt.show()

## Definitions and terminology

| Term | Meaning |
|---|---|
| Cofactor matrix | Matrix $[C_{ij}]$ |
| Adjugate | Transpose of the cofactor matrix |
| Adjugate inverse formula | $A^{-1}=\frac{1}{\det(A)}\operatorname{adj}(A)$ |
| Cramer's Rule | $x_i=\det(A_i)/\det(A)$ |
| Collinear | Points lying on one line |
| Coplanar | Points lying in one plane |
| Determinant line equation | $3\times3$ determinant equal to zero |
| Determinant plane equation | $4\times4$ determinant equal to zero |
| Tetrahedron volume | One sixth of an absolute $4\times4$ determinant |

## Section 3.4 worksheet

1. For

$$
A=\begin{bmatrix}2&1\\5&3\end{bmatrix},
$$

find $\operatorname{adj}(A)$ and $A^{-1}$.

2. Use Cramer's Rule to solve

$$
\begin{cases}
x+y+z=9,\\
2x-y+z=5,\\
x+2y-z=4.
\end{cases}
$$

3. Find the area of the triangle with vertices $(0,1)$, $(6,2)$, and $(2,7)$.

4. Find an equation of the line through $(1,4)$ and $(5,12)$ using a determinant.

5. Find the volume of the tetrahedron with vertices $(0,0,0)$, $(4,0,0)$, $(0,3,0)$, and $(0,0,2)$.

6. Find an equation of the plane through $(1,1,0)$, $(3,0,1)$, and $(0,2,2)$.

# Chapter 3 Summary

The chapter develops the determinant from a computational formula into a structural and geometric tool:

$$
\text{minor and cofactor calculations}
\longrightarrow
\text{efficient row-operation evaluation}
\longrightarrow
\text{invertibility and product properties}
\longrightarrow
\text{geometry and system-solving applications}.
$$

Core habits:

1. use $ad-bc$ only for $2\times2$ matrices;
2. choose sparse rows or columns for cofactor expansion;
3. track every row or column operation affecting the determinant;
4. remember the exponent in $\det(cA)=c^n\det(A)$;
5. connect $\det(A)=0$ with singularity and dependence;
6. use absolute values for geometric area and volume;
7. use Python to verify calculations, not replace written reasoning.